# LND pipeline — full run on Colab (SageMath)

Runs `run_pipeline(m, r, max_deg, Q_list=['X'])` exactly like the local notebook, including **all** printed output and both `LIST1` and `LIST2`.

**Runtime:** use a **CPU** runtime. A GPU does not help — SageMath/Singular Gröbner bases are CPU-only. Free CPU tier gives ~12.7 GB RAM (vs ~3 GB in your WSL), which is the real gain.

**How to use:**
1. Run **Setup** once per session (installs Sage 10.9, 5–15 min).
2. (Optional) Run **Mount Drive** so logs/results survive disconnects.
3. Run **Smoke test** — must print `LIST1_LEN 3` and `LIST2_LEN 1`.
4. Run the case cells you need. Keep the tab open while they run.

All output is also written to `/content/LND_results/` (and Drive, if mounted).

## Setup (once per session)

In [ ]:
import base64, gzip, os, subprocess, sys

SAGE = '/content/sage-env/bin/sage'
READY = '/content/LND_ready.sage'
B64 = "H4sIABpV22oC/+19S3PjVrrYnr/ijLpuCVBDbL3aM8MZuUrdkrpl95Ni26IVGQWRIIURSdAAqMcoqnL5VrJNKlnlViZ1ZxZZ3XWqssnCU8kPmV+S7zsvnAMcgKBEtbptacYtCjzv871fcJ9/aDZ33rTcb3f2Xrxs7ZNN8iYc+TX3xTP3+dbzlzvw4Oq6tt96+87d2m3tNN3dvVfwCx63oolfqz0iW5MkXO77Iz/yEr9LelE4JK/ebLudcOAd14Px5ei4Rh/GXt+ve4MBCYbjMErIUo1/SIKh75D4MnZIEnkd/9jrnNbGUTBKrN7C/taLHXJ15kdxEI4s+3rBIb3BJD7ZxPntGm+38OjRI/J859UrskL2W1vNFoEHmaZ0EUHiR0kYDmKxinEUdiedhHjwhH82jLnzZtswYq7havnk7ASiIDmpD4O4I5bQ73QNI1Wccq1wyq7fI8NwFA4DbxC7Yc8994P+CdyR2/X7ke9bTYewR3jwXtT3E/zGIUPvAj+4Yz9yz7xoc33FbtQI/Izg2pv1EVx2bNn0Cc4Bf1pB98IhkT/0glEwgiH8izHvgz9Bj0ADsrkJIyyT1fQL/DmHQfkyDqHVkfYl9DwnX5IVvQv/Rs5H/glH2TQ1wx8fZkjbPnlCzo3NYESffFk4Cv5cBv6gC0cgjtVasmCn5DE59I9sW+vmD+jazavSFl+88OLpVrLTRX4yiUa1qYdqPlC8cTylYTCy9JPKQUM6rT+IffMwK/oGf4NPsHV2LNm5F0Zw9sGIRB5Al8XGeUxWbX18dh4UlTjQYSMF8AC6fLKEq06vpZYeDxkEcWJh1xUd4g/xOPOYVREF1wtR8BFZvvUPDLJaJ68ngyRY7kdeF2isgAfCyS6cnrW7d7CzPacZc5RjiLOzyXPEwz3zOxoBwQczEZEf4MkALkUfAg66mMKwOW5EZeh685TGF6xPfXjmDYKu4HXqFwiwpynA/mAbcfz88PSoBL1VJMFVYevflNIfsZ5duBe/sNVx5HunuW/z6Fq2jn9ii7+75eBPxxt1gy6IDSqBFitA6gO/i0l1ENMLK14eDiunMHOGAR3pN0q7+e8WpmBdYQq27FGY0KXzJ2aeMwOv4cS/VgnYJa0vocUloC3A2siUVT5CPzuF92qreMv6/TE7ZIapVeQSI//cVbgrSSbjgW/l1sFZBX7MbdnObhnEViuiF0UbR9hYm8cuusA8t9L6aZyqkFXRHWTIYwHXWq/KtTYKuVZn4MUxyvANSYFdF1abuK4V+4OeA4IrSLJA9GHI/iZCsrJ9bFFnDeDw2Qf1rrGPGXdpz6bsBEJOfexFPqy6TOiQvXBgfRUjzljYeLb+JcA8LpB1r2dkWtftwJXL/fZ0NtMzEHV+bWw4ayWdLPJj4J5yJvUrBKXAAepA4ckfTYZUibLk+jJQ1fWj4AxG6tXpJy8JznxLk8n4AllDM/Hm63m8yVstqTd2GBzVMjti7eXJoAIl1ggH45BTZZFyrz1tj26KW6d29tiU07HYX7ntiEbG/egE17zoUTAYh4k/6lwCHHf9i3T1SERwS5tPV5SVdSZRbhOnOuXBTgbiA6ulnQuOni5OZw9sLrp9+Ghnd6LOlV5CDPtwT/1o5A8MIKrAotWz6WIUyI78ccQhO9+ntwB4b10tOmSx/qcQyHicRIA/fZsB67CPp6AAjH1tLxhIS0Ua9LRcefUmx17nROxzW1FXt7kYKuVM+Mw3g6Rgu96kn1lP9/iSN2HWDAo1Xuy7EWMQ79/XxCV39UvmvfRrTiVjlAYqa9hdjeOhCKAMpMNKdtmHXRAjgdzrcBOOkmCkyKUpf4DGXVi0OCR1XqVRjt2ap6WMSFmqXb6KtGUyy+mk6yo5puQO1huF57F+uAgHQ4SDwvvZHiKMWcOMFBai3HW4PZQim9sJ/V4v6ATIw5Ihw6EkM3ZyZBpFwqfVYd06FDLD81zruO6Nx/6oa8HndEGvqYSXRMGFJYdyaPO0zdfQ5nV94PcSgWLpd/xsx+Hg0nA8IH/gcr6uw9gB8E39eLATkrTJ0OoAexmy9TvsUP8cjC2mICpXlKP4dAgz/1JXJraOf+TWnoELtZ8qaGVbG4hURWr2xV3bAdYydoDXjECCdNoP0ao4nKP6r9Fe1QCQ0mGu+Qsyo+n9/G8DTRbavtrNNtFkdklIiykEMqqMJw6IhBRZQVRtzqMS0p8SeiZFY29h9rWW2GTVSH0Fk4g0ZFSl+5I4GWg6Xa1QZvjfqLs81s6/QJtRV+CNLq0EiT+jRtgwo1lMo7Ks2Qy0tgpzmNnEdBs+cfMd/MLYxQO/YP3VRtWBR2MpM/GUL6rylN+WSsi9YACqgYsWBWkKjq3s7A5p8hNfWFigv7/2/TEJR3Bm0DW1IsfEQkTi9DAmIRxs5J8F4QQ+j4Ay1rVRlH7yxnEpBnFbytZxGIEEmFti/dS/BLhQAAMH8hJKdzPANMZxDDw+p4iNUfNZzUNETgak7WM8xiJbr9XvOgSUIJg53bXR5svdHWjZowtt1oOu7w0sk4VIm7fYrGg0KbJ++THFuUlQt3NN0h2IRhbsbmzrFJWPk7etZeQq0VCFf62ZAaYrAv/v7lqgWq+TXYpCDQAogRJvdr5Vjmh+IpURWTXJqhhxZ0A1hmH8MbVwlaKcAzu/3Bx4w+OuR7oNYiEZ7tqor9o1E+6KkR9Q9VeAqr+riqq/n51PTQN91CWCkavYQQXjkc8bVH5DPgV/Bd7xAADe8uv9ukMOD5y2892RTZITD2TDCTTDE2SmVi89WMbSGBUALjeRsuMyeQaCEDNX+wN/CEIZ5Y90djkfH1RKScvkHUgDXFKhsvWJTyhYkTRG5vgSHQopb1WozQ356y8f6R+R/dNgTHXEqXcSjM7CwZnfzYmJKN2g/jOucxXjzEZHFpNRmSapQ1yFZT0/8TunbDtB2Y1vAZYU3fh0yhb0SqnZHnWqM/J12Gd00M2TwSPbRAfphe3NTPM+b8r2+8oxXSt3KoVwSWSjTnbi2B8txqk1h1idMIr8DgKQohoS/wLD5JIgHM1xCUimfVyBqklsG+w3Bw5pO+Q7+gQxhZELgDWHnNCH0nvnpH+sqn+sMXqxC637SwdAMntL7ZpQMJv1Ht+c20MHKtcxW/BVSlmbqOl+7ZDF1qKNrjve6MTFZl9bJ3Y9nhzH1tVBg7SewMLaDbJyzdqgskabQeP6WI5otdjX9JyRgLJ2deXcYysew379TYoKanO3mVJBqXyzrzS/YQfIDk43wtO1bAP5E+MJnOjUuVMQ25d5QXMdWWN0Yp4hJ4L19dFYJNoxfxDeQKdriVasT98FFBgic+mTJ4ShlCefHXbwYXabbpPt/gW6zZa+gzs9WTqgT166sAMci4+wBG0eU12fj3kYHMHD3aUlK3jMjXpBar1CSyFvaPMDeAmjsVGfcBcdR/jDXYe8cMjLIxMOV8X21RJs37zlDwzR2tlvwaWTbevA3lwBpLDa9uYB/v7O3mzPYw6+I/g4ZTKcasGuNet/pKLSl3kMe//eIWHU9aPNRUpFzwb+xaJd24aW6CU8RGe2QwkCMBbuyoHvrFUH/gftUjFlpcZttPD5aa3GpANqcqru4rNrXH7EOcosH6g2yZt9oxs4gBHzRVmAh0CQJJdGmSL2E7sBZ2J2BSp+QGHnEisCIpQg/zrkHJabpBT0F/HMhGyz+a+61zhVTnIy2KDSvvhzNb7m/XQyYJyBSke6kcdesE1gXxVB1u4IQRTkeBkOQ1wvSkoi0iEcSXZIUGfAmJvbzvjJoIqQJeMElFEv6oodaiikSZ3bKEvGfnQG0n4SJgDGXM756Hh2a1wRcKtC7RXSfdrRvpaS/kIRblGegfGhYz2OhjZzcjEaKaJcQS+crRSjoB0g0cg34k3lIOHV9bvGG/L3f/5//+XvP/38t7//M/z3t3/89G//96/w57/xP3/8F/j4H/7x438nzzkaqcDlsM0yvTTVYH8xGJbFpEeEE5NGNVT6lWDSI/KOgoGPbuUUDhwyCjr+4BIxbegloJB8CrhXOdRxdeOOcW+twbQ3JSC/E47iJJpQTWZOWPTvRhKP1jgerXE8Wud4ZK0fLK+17e/XboxPPdTBrDW71qcf1u3arutfjCNNWUNFjz3+fi3FQK4EAjByQRy+MKiU8n4yRxY35CX3FlA5vNq9Vp6ganH1Qn2C41+9xCfpCX0D4gLoa1xgQGoWn4STQZcc+2RFynYjDzPOqNMQgPbQWtgFsNgF8mEtvIBPL+inl/Dppc3NVtvcSbmt+A4lullXOOC1jeuhDa/J1WJKjf/x4/9k5Pi/ZknxIqIgH1omyUztiU9+/svitQknKoferT69Y5xYb5D9AVANTYJLTvw4iIl/4Q3HA9+eP2Ksc8RYzTCYg1ughMJiVhmLgdGyTIYAKq5LQQ6681XgpzZFVPwEK1lXWdHyKsmIdV2/E/le7MdCTwGNZVVhRRufmVT3LQ/6E9v5vMS7yrFfq188oNOvC52qowgb75eJIJUDWVZ/+xEMawYkmaNJ7a7xYr0KYqw567YO8iZoXjeA83o19rA+hT+s34N5bf2zt69VjnlZ/d1HwJMNxbVkF6jkAHt3gzsbxRPeHHc2cmYAhjuwhee61bCR5yarkpusOZpVJMOTkC1NN2tvGHBvoxrubUzBvY17wL2Nzw334MpVLdCsG6q6JqHwSJW3DfhwLU7J4WfBcz3ZYQSJP5QRrEUbU0eGLYmxx/ZsiiH22Nych1pYOY5n9fd3S352mKhKNurCFOhxMnDMyUBnCQnBY+ItteE3kz2WT5hHIJzE9px9ZMqCUurjOcdOx5mRBmVNkeq/sEnYn7Y1JE5KeDu6Vin8IKHBLii/evLJKn9yLJ+s8SedXJuDXBvmXreAdT/lT76rHdXUxAGkcECQl1ftmp5NQUkffLO6YheKyjdPD5mqlBqCxPTQsM1DHpBwVEgYJTnM0UAeJSV1WEN01AWPjrqAxxciQLmM4EmaIEbVQp9M5M6Ag1WL/6x8LGwVBscPHFu/UZj2Y9L+fv1uMZWJD8pyUlz94HwzH0yFrX1DxQa2oUL8XE/x6gN7spbi3jeyzbqOjWs5bPzCIb+9ETY+henWH7DxI2HjWuXYkbU7jh1R3eItP06K5OjH844jqT7xHN14B4iGKRIqMjj1l1d2ln9KvvJZMeKXGn5SWUQvksNvIYbPLIXPSwhfq17Tr7yoXzQZud2ey62+1tAhUUHe7cLCQnMyIkNT0nA4AnCGzpFdFwHn4rwYt4Vvr4bXzlUEBweNT4G07zstB/jtEUdzGhqKqM6fz8CBOWZKM10DGO+QqKf7v9lZ/qt2KT//1d0HPNiv1LIFLVuVWn6AlgffR5XaMg6fpVffwy3sO6SFH6MjucE0XRt1cFoLCMPDmd2etjGJGFk14CB9OkxFjX3tqRAuWtrTdU1QoU+jrLDCIjcLxA8ll3qeggY9nflIG+zUUXgTxy7AuJTMkiuUI9DMfman8f/cVH/mDSY+5iPmjO/mRMbKooohRyKXGZFSrUMgoUcqxdXSS01koyqBKQ5GyhCXdcpxrVUQXzdMNRIrx2CsbVSdcoM6f4qnrOziXns6rykruwHXviil3DGgz6DvH0eeO/SHx34UnwRjq+diFhMNnmiq1bQEWWYZJucnPpCNiPTJsT8IR32UZSgxSQcFGs3GOiI6n/pJJ2PwJ9BtOvYe7p/l0byLwnEYB9Sz+Ly+Vl8n1pk3AllpxIJbePZvk8IgoBNuyyHvbJZKB5Sfb4ShUxwOffLOYWzToiHzDq35lUkihs7NfDkwDKjv69W/8nUjL1wMu4hFFn+zLhKA2HMRMD/kBSf46uizS9n1sLfYdq+C68Vs5PnQPhJUHAkJzTk6D5ITcrAYA72K4oRYJ0Dw4EoACjBj4xKTzeC02tCgQizeTz//BQsQ0nQAWAoOC5O/eEYn3c+z0mY9zfIHmkI3sCnO4LHYkmS3jNHSjA3c5z4vcXbYGDGC39aejhpyt3s0d+krZAJtN8CIHzdgA31Fe/AsozYWC4Nv9/nBwl+24QxtmRfwVb0fhf4x0GOXlwGQ54tRZQMaXIa5fH0yDLuTQUi+YhePIUn7Vt+uR3530vGtF7Jjv4Bjp58VlABQ69HReHpYzNJ922nuYE2p9jfqiRyxC1ZBi+7tAvd2EOfrZjFkGGn5CArQmwhFVZJSqQTAWeC5KSGYkk7pkDM/Og5jn42jU5tdL064xJgEANxsArFh0L1izMOj4W0ekg1Mn0XAXNay8njzNd4ca9IFZ0EcYHoepj+OLkkM7eEv/wJuB7uqxWy39ltCN1m3VQqXkk3SoWTR2n/19luHHE8SEdFpa9uhZBd4Ncgsl1Y3Uy4WloC7AJjq8iqLtrGqn5Ifmb147An/ibzJu87ZFFvh01F1luuwSOP0MVORJXCY1KLUGNRbaufCoaNQLGHZskdKxALIlVfB49XrJ1dyRaA1kAVDTmFvQdZaxU3TSAdDtqh9rWinHz3DfF6poiXZnbOkd9IsP4aZFOobhdiUIhEm9eXOviAvtLAucL+OE3WBmY7t4sq4FRLj0+T4gs3tD8JzsTkzvps2VJKDyrExLUlTkhZr7JxElyVbHnFy6xCX1q4xCXZyeoeMqQGocLReOmDxnBWP2r/o+OOE7NBfKMx5MSkpawwMv0f2AXomAw+ovRcMYLk9jJzF9zCgkLmAuACzLpSMYSFD8CNWk7RBvLMw6MY4DEh/TKo8UyuwZn/GXhzXPkp28e3Ti6cRyqzJrIqliRYdebHzZqe51Xrb3CdekjXQ5YieefFmm53QIaunR69VDlhZ+90cVR5NG/hUtID7ksfvROQukqi1YrzTBWAugDf9GJDMJ5ixfcYLM0ADQU5qOTrarLPzqpkolWK5RXqgVf+9vcRdOQZi7fflGf/E2rDJ7gQzpYIxaHAjn2TLJqKiqML7tNq1TinR12V3xSrGU9Ez5oIQdfrEl6trSJtvRisVxgDdgP3j//j5b2Q/ZcFMAchp7coyGpkycBUthUV1fA3OhRs5Z+hYKniU6UqZgjMmoKj6OpApBSOoC60kE87iohzbmE1hS82jo71zreabP6eYZktAmufWOZg++JQeIK+6YwLSir6cSs7cQj9PsRt3vbIbd73YjVsQjQC3obpY7FsGEs0cR5TxIcC1QFuLXg1GDGc+rztPgalVu2HNpG+tOMt42xgPtErXVHDh/JVX877vavdc2cu2vlbtntH6R5FORzmn0Ak+T2SE4T4XdKx2PdXfqrR+t9EU+4k/JmvUBtggrfOQxD5ImvgyGhStMPOZkFcBpbUihUPzJmHBOPxGcYvaaac1+kqbAfVtxsQbhiAYJSf+kPVr045td8Wp1+sOSJ7D5dXro7lsjIohHRABJgnIpmIFUuDOVgJlz9M6amO9elmT5hjlt899vw2uKZ/4SkcmBdIybJnTwbaqDyGdNj0qWp0NtSE6ChwSAHN71SFwUEfayulrsd5QYRsGwCWEUdAPRrCAtBjcYxp8kDXxVlQZxOtDePOqesQUZ8JanTwL+nR7DXo+igcTtqpsl7kbgDIwUjF/PSSjhqj6fP4smZKS0VHULtmT5vtdrzOPQoO7FFbQo7CKu8QPa3Sr8BfDAXw0NDkbKio+fM6NOnnBFR9CFR8EEeZNePGs1BXxtK4Ux0Wggvay0hwtJci9B+Ttm1dt2iILXY+omx1d/KAXRdTZn8K3ULQEuEu7VJ85f4T7oV+sfKWg9EWdvA7PfA17khDwJx7CGLD3IhyiBKgKGGWBR2VWuY0021Zkp+9Hkt8daUK4fAzEqG2i+lX5w0aVIBnGsvUYGen+cJlCoHpB0hCYq8XNxaUvVoQZWnyRjYyp6Ohj61hmooMSPiOGzcx2s6iaX0lQzb4qhvH6nt4pELttcpJGKirRNSJ2cCj+H7EIQo32alEwmXo7nMKxcgJBD71egNgrckEoSqxKNduQ3gms7oqv5drRprkSn/F5GpB4xT8KgLhhdKLkuI9iJu50wq5PUbQXjLpaGU3pR0kdS7wo+7QgmZrgy0KoavB6qmSCYE/ytnzpFtPDf4oV80ygT7m5JH1jH/dy6uhuowSh3tx6AwQSRpT4YTGPBN/LcyZMScLFi3vS4vxA2vkblFXZUHgqcHwvShgf90bEH46TS8VFE/vJH+D64HsArD6W8g/oi4jhEV4P5ajifup8xFd7tNaPWfyUi795HUh1loZJ7s1JiOZF0BHg9MxCcHaQ9tzWzg1OGT+sisX8BNkR6mysqJvYDu/EV0u5KH2PY5FXhLJ+Gjhy4gPB4IMM0BciILAFlIu2291rwrf8wPmtox+AluLh5AJgw1NrzAhHqgAOq3WCNWixVRxQ40QMI9EaNH8gWCY39aYlqNFe+km9QlzbDMFrs3pac9elOHXmB84ClblepIlkCjozdW9e06aixCGb/0iSGQUZNAe4omBdK0iXJv6b2+KLKddUA3kOmh0GrHm9MDcYC1Sf28mzclK4z9a3b5lmLRXtweV8jxpOegFkjoXNBZCkVPFqgaN8RVnt6xzRo2ivUTlNfFNn5NEWmUoMhmNWqzKoFRmu66qzrnBn6Clrqy+TzY/GyU3FXaewSXd71Wxf22VVIW4+OqWIuGyQ04OR3y0+zKBnAuZG9gWa0EY/biMGlJXBwEOHTtPKYFAGbmccqYa7LcDAWiaegyrYIGENveiSGMgGbXBEWlQWy+R04zkw2qESDTVKmQOh6dyy3mF8qQ1Rg43V/al74OyJ/0jBL13zl19+SVpvW1uvFG82rJOKk9cFMdSZhC8TtzFmb2WikjJ7kHYi7ZocLtnyexDjmrTLqnro05vqoRlTXEaLIDN463SpIU6HW2somDfTeBZv/Z8yrW1FjssZ9EiSs6OK62K81SRRGqir2keTIdMf1qe94rRX0Yh6pEf93ZUaL+7msXKspcq8qk/4CZmMmTyIQZNKISeB+zdX+6epztOU4Qr6sNjLrczTXB5r7bzLKsxzGDgrca0epchk8d1vKnq42N1mqoTbt9e3Z1KdTdS4Op5yuprFfwSzJJThmYopYX5XWKIhlmiFt57YrN596irMvFBGI+eMDnMX1QxAw9XQu0I6UHOYxQTpXFb7r67sFKnkuuZ9P3rRHK6zkl40p7t50Is+A73oV6MDGSXoqrJ2eXpddbd31qYr6BNrT84Cj7yI6BX+x3/8+N/Qc/EvcF3/izoURc4WdGN2t22/ByoEjXueYKVJ/2I8CDpoypVv8kF5L2KxkjGTAONE2u3Eou7VLa37pDk+368/2uhxNmWpmVzKt3ciFwIA8yhXRHCYoMMMjrJeRQoWAlI4xdciZc2Oatl3czHmQbaNOOkumgNqM6H/irL8rReNoP8RUUeh0f9+F/R1X82+meIy1/3lMzrL5+QJ/4zc4OagaesY7fF+D7px7L1x2PQMzvbKSYjrv73Dl6BhaRt/MIaLAn3ieBIMuizyohz/eNAPvugrCZJJasqfx4qQm9CVuP3jHBO5Rzo9L3LLIXFpaYnsUgIlE4AQcRAuPeomCTi3goZs/rorCIZr2aqrC8ssY0Yf3okcTNo6Yh89zAkH+dpN0pKfh6MzP0pU8ZOi+b5IxpZ2J3R123QjmIXEOC2qbr1JjMVmWNhbMHLxMA+BGzBkxRQszv/5nMbcC9YTP5ewjh1J10k89jtBj9HGlPDLxeHj40m/j7U7ex5mX4y95KR2Ewag4v2+A8cL1A7+e/HMhM1V8b48uSePIg55Tz+a6hg8uzVq11OL4zu466GPL20V5I/LbmlQYdYmIThak/1qmOIWeZP32bFUhzV1WyBkw25FwpJHfpiECX09Iw5Tz5oOAMugTQ/tMZkZHaAH3hn17eKZKVtsyReJBnKTqcCzKwSeXRlDN8S/hnAF8NV78fR9bGdTN/juclUX+PPU8/0IncZyuymXs+hGIvoOLrnc+yGLaYRLNqyyfb+ZW8JuowIxEKikQVhVhV03qESKaKo4DiKCZSqRInUJ2/kLBIhuNuC+VwWQuIgjPcwkZjv9Qc62b/3AVvUDTvBeWZUkzMNjzIE6DkE7wB3GOeqprP8xH9rODEHVsXK6UI0qDoJjg2Q8A3GsnCe2/vtPkDjqBCJPF25BE2r3r6WiwPK8udfae771Cv1C2JW8u0xOQuX1zynnP/G64TmFfPxHNljcX0wlGkoG5k4jaEczodC+kgIOQ2axmEP26UYEgfYsoQoKarNJTPidojBb7hQ8LlURBdq5jN2V4V7lJLuNlY+ikKw1xNuWVe43nkT+cluqujYr7Ab7mZPmIadycWAkIS+esXOTYCd0EWh6I9U5tuesAjfbBapBPiqOmUS4liEkctyIS1djUKB5hYZUqcWmRpV2o3Ka38bqR4Gg9UYu6o5HxWJVUm6x7CLw0Pxh2dSlrdS7B9TZ57eeT+B+xo3FMc8Jq57ELerYiHxqm5de4OPk63eZTrLqma9NqxXAeJrvtlmC2y6asR3qsVrF+9Ysua+90QQ2ean0Y7LVlyhcEYsXoaAktRP6UQffmC6LXcpkd/SYp+QZ4FgpLkBfc41vuHbIMORCO5zdn4MxLkx/O7hNbd510U53OoKiMqRz0S521nrfPtON9+1sVRkWFS7GFpfaPrOzrkts+CVZyfsu6QqWNtlZ0pduL2FjxTtGz+PxJm2p4xt+Y7rMqtd+x6l8ux9gknd773Ze7b3ZqWqY3ff6/mvUtR/JIAHuywb4UVCWBY3jw5QDwB9nav3aeeXtJaF7HPQ566cCI+g0SJNz7gtmC/FUZZIyoSYzivBuWBxo6I3HAQvYoX0NWXK8kCCzo4yUL0DW5CMx4XKLilUYWUz9G50ojGNW90tZh0C1WPdu5Ct3HLgeRUAxhSIt6RgqGvzZj0IhVOiYKZCyl0PJXjlC8qEFWmp4+SeHHhYWBNCxs8mXmsFR/2IM9HGUMEQVOCqGyGGqaF6MqvSADv9EUVW0rhUhrAFja7fSQ5gxVcyGsuAfpeaKI/zpS8ZyMGET5ICgE/Yjb3wSdLIWjjpb21sOeMtSDu+E4wCgrIOlIinwwXWFg0F4DgzxWKvMV7+xupIHu9OMEqLGELDutmE8quy8V/QW3T+am5fvDR4seosOWTzGfzr4Txf/8RePpFowXQM6FRqQwOuczJYu5H2aHMMkN76SVGeR3yu6C33Gz6IELymbNH+taDJAw3AzxdTMrNT8YOr4Q75jXlHRY0Tk/uhKhWoEI8JHnfGqJ8xqIVO9BpqmCpDYJtOAsinQNzBHqIjKB3fYmTtsxVQ1qpVpAqxVqgwU8YkdrrtwhUU1Yspa1Io7u4r9Ba35QyF4dTgfokk93OmWTQQHgUpbk0njUPxcquqRAeSq8NtmeoQuyQl7BD07eQmgSHCYk4NqmpRaLWyKRsXGFe3xmPrkNxg2yWpsZ5Wco5SNRVi7MOgLXQc/Ky499t5h2IzK/Yo5YKZ8JR05yxSNbRQ2Kb/PMDghxLZz7FJhmW2xl5Rf5imFoZigmXvS0WxT3b9i9qmyUISFQg4qz1fjocoLFOgFZiuC0iuUMX70bdBF2mmtXKljaNx3UjKQjVOhdUeqVH7mdIkxTqYuMYsbq6SLrdkbyb0AnfP+CBNBuwzIOX/eIghhsH6sggSIBev2QNZNV1+HJqNwlGmFfv6aCI6EZ/myuoA5cZgfTdton+OhSvr7GmVLFV/aVtV8Z1Z9NRpAb032dmiF4JwGTIMNazfRUpmVMaehCjWMH/6H2E/jhqSqqb5EHhgVhQI475EiefOjZyK3MA6fn4QD1dGUL7ndFiIRgjjSRboeta6xF8CSvsFA5Z0oCiOdsPQWdi7G9PWRLKST9b5Wb/0PmeqvvYV+mGD4nJj5up42sDMSf1NQO4NeXiL+304nn1kpLyBZZ3ZtdllfVctvIusbdO6q2nlxIQWAGJHDIorJZUsp5DWJmXMvqmrsrObeO1FzL827kPqAOcOJy+aWUvGHvgqGPb+ekrxxTzkZhekXMjVdSRUqSjlNEyA+y7SGyDvPpjZkUmroJutP4SRyBhuurJE3b1sYltklmIJtFx9V/emRaZTU082PkCLq/WcdQHu6knzzvJ8FViuNAIJJ6ZiLB1p4P0wR+M9wkuR9zVCQWDBOLIRcZsAeO3xtdkEdZ9bZLNAZD4FupqAMcDH4TSnDm0uGvB0wi0QZjU/mkzkNgK1lewhw3tuvAs2GXHNROIIG8XJDhgLSa1OyK9jKMxkVM4Ian7USsH3Y31GgTTGRgEBkXTLac4n41FbKd9ON8JyOSk60edztmnK3a3C3kRLSn73S9QZ78dkljVXHKq0jP45npVXrR+ZRcrTq7knBMIhxvXdHDoloWEDtzMTx1kSJvwGghCixnWeIzy14HoIQG5PSB/ywQOvsshOmWsgNxt1vvX1Xifbp0Mryk+6mvsKLbK5X1WwhDrC3zBCaexqQMWeHUoPpWTqcYt1VXg4/MUkXUQV8EOofhPrPVqhPWdyDJG9k3w8C+n0J6Kpg/XkK5r8UcXs+ufb1p5VDSXTrbvUNYZgSigNI7sU+5rKHHNghTUxD0QHqbr/QAsSzuK0yPgnPR6xCIDrDmk+sA9tO74jFGWhRzENv3Jh9XSKprk1OvFhxOijZfrS88CmWF2Z1+EQEAEMd7jPE1z7kvIa4PhfWhZIFa3IYHDVSY23Gi5wOal/X9MxADM1tqkUFH6nJxz3kUNphZFMs9IGGYVeONWWgx+Rgc4VYIj1FsZ9XlEdz78F6RLbGYzXsDqe52b1l7NuUjNRxXEucvG1u1+RgJmvXwJ1N4GqDhL1Gs1nLLhl9UKL+H/eKHdAyrxaNKZDHk5/OZafN/mCruzpokJXrjJ1fvyLB9fg6S1rC8HpjnDAzdorQJC1WCrpBQ9MNTKx1byTrtBNsjBNMa4+42lDa44KyfeSbedjxbJpiDmd+DdY33iiIT3x6hTKhii4nM7uuBU2f76ef/yqsF/+KLJQ6MZVJlOErvD/t5rdxqOUDKfnWRzNYiyoIwXdqInqw/PziLT89GiLK6tI8WIF+YVYgJ8OAnDxDejAUfeqGol+9fejBAPRgALpvA9CDnedXaud5THa9DkDFJ2utQZtLjy4R+i/IyOiPZuXIPGdLUfUFs0Hj9tYJzUSQNSboBEhZFFfu2RPLzloPXiM1mAzTiL+wRw6IR8Mw6WthZIShWUc2BBIzTWLkohECflvDoQggPGBoOByyiHC6MDWC0USb2UjGxD+aWgFqrtwleUKsA4wzpJ3ML+W+cMfhuR+5QBbw/SwLB99f0dbXZIlYVzgcCL75GG6jVp4fD4mNyS5i7i9OaaVWPurCykLtI5tvBASJ1uLv+zP3NL1zae2pYu7Z5Stm5h6x/rIeB8v00NksV8oVPFiV7s2qtGs0JO2zVwvUsoMf8i8oFWO3GR8t2Drpjd2muaCo0BlVKsyuDXrk6Vw1GgcIZyRx+RsR4y2Y3yvGruX/LBh1QfVUyZmAiiRkOhxJiZyXMOOwySj3YHN7sLk92NwebG7zs7kJrltgh7u/GgmdMIpolo9SJKFSdYQ0R+oJ29yTURiBOh382eefluFehh+zcAKrFsJTVmP6dswmLlWkVGPNZ5l4de4NTjFj8UbFC8pqF9xd6YJbVS6oWrKgYq2C6kVFajWXv0DR/XZn78XL1r4wfbgvnrnPt56/3EGt9HqWUgYVywNUqw6Azyaxj+IoVkjeJNYp2mBWV4RtFbNzrWQC/N5CtSZfsQtuMv06X2sLvk7Hl0QMx4Um8hCK39307MPeq23Zbpucbl6dXhM23OZVOvR1JrcrvSJ6H3KEQ5iaH46wDZVVMAASX1a1gG8nXUa6kdT4nQMB6JF/lpb050WJVh0btNFRLa2RA93OAYbhEjLVc5iJazK0fOhxThfqO/CB414K+xTx+MpsLqMyHNWw8Sij39EjHHoXVjem9ZNgQrrGFcXUlu6XgYNYag5eUlTGHP3Y6/u0+nNcVxI6EZ1cJrUGwzFIf6QFT97KuuwyPR6mk99Yi+eplyLdJr7Rt6CJXLU9Hf5ev93ebwn4OzhPX7RD2vCHHMkEhzpnFevupT4V62p0bTvKn6fX9uL0Je2/2nsNwjVdkmHawmoa6NZhNQe0omf0X/uWNTI+fokMcyWLQ0N1DAMGi4IY3B+0guAbDP06/mPZZfidQi/WwKiLKhj13mTEEpo54MqqceILB2TgY9SP5TfwtxxWfWgtsklxhsUUROHhPqXVuaFph31ZFVRW8KDVXdhX1le2CShnK/QxCIZ9saJC6Nx+C5JYIbmmx7x5pRz2crLSqK/1rpE/bVIN5sUz2wTXUviwCuuLsIY60ZdGssJSQjcvR5KpRKJErGE9EazeD1o4dTC8f4+F+jFqjwftHdXLSodUKlYYjBTHbJXyIKwdw7WiOjdq9ZBZyy/SWZQiIab6IDcrD1JeHaSgIIipCMi02h/m0h+ZjHlKOQy1PnJFPswlPqrX9iiu6XHDkh6lFT2A+kaXVsT0Oh3HUpBXirRly+8w2ijYD4V8TdWoDomFAG0EjAKgyIPDlIoJ7QKloxIATLv8cj3DeOm5C+c3h89vVrWDa4u5EFdeW5LhIsY6Y3F2/grz+u0KbJTW1MD6n5nyGQt29ZoZd1UyY+aKGflSGdWvXiuRMYuKORcDyuutvTfSgDKPIWeIrXKw3ORxGPuMyTdmroKXNcmBFvTF0ylxWf+eiLVsyldI0qd8YSKCKvcC1HT8W4ZQvXc5FTtsmopUKydAlS2x2dtHXvUH4TFgf1YHZTJT3l7BBysL2tJ+TKl+098f/wuP6zKdkDkJsNpp3TAIrH/sduAsfWZ1ukMni9m3OWOAGHtlA0qawRmodbE1tlNJNMtxps6TGrRSg1U/N6BpEdiLenBG8gDN84pvhcIxazDbM2hw4AGn8bjbTBsw77o1lBUbMw2lmR+cVhcbZ6uLmePpKlQZ80qi6D6ZELrGvcTQwTmgq8zd2m3tNN3dvVfwq5E1SXKQODwqpxJaFmYl2lAYnCeWpwa8HR45aSzes6b7Hp4dePRXm/0CJfo9t1qLeCEATRbmZzjFkjcFpnqxmG2XzbbLZttls+2aZwsnCUw4w2zME0OoJ2ZqqKG6qRnm6BjeBayGYSgvXWLH69DTdejhOuJsy0IaU32tSqyinKBGqv3kIyOh9xGnIe57JXDn5hehONSeqLdCgLB1TtXj2aWr36XHs+sIYJhGRT9KPCZoP2VYirw8DaB0ePCdQ6Rj0NF2bkTkGXy504NplHvbqitLqxBaWE2ZLKTU6Vzd0litGWKG6LAfnu239lofWntv35Ddrb1XO9vZV3lqTKaWD1PMCyUzB1vtdQGtg+SSpf5aSQSSCS0WjLJBGNJ3AdpTl/SIPKurIbbG+yiJ36wU7iaP3hT2NuPh7249b71tZo5d2c7zOpBsDulkeoGl3b2DBqkPOpZN5ar6wPe6GEujqOrKLnMnM+ikZ4KDaF9KjOuqIaHQBcl/hxrImEKlWINNW5b7UU/zKh39Fqf55m3z9darve92SuDYsA3twBcFACkti08M29BADBgsbT8doNJDeNyTsHUlRys7g8z25btb+VvbBMBQeogvu0KJFD0FyGa+3Wu9fPuhpZRIM+8oL/6mmxM8bDdzrj1+BLognG2WPwRGuKmGhXGeV6PeLSCgubP94fnOTGTsEdmpqyofZZ4N3M9wEifk2GfvX6RBj/pbiFlvLF4A2pRqp0V2v0zrGTMrX6bq8unh7ip1VewKuwJ7I4KLhM4FQldFS8CDVUmx0v22pDiluMRSlq2IG/ha1CAa+l2VJOeDLmeZmkZi/oXWslNI/gzixsg/T1UG0vFG3aALXD5z/3Curn7S9CVFOij758rr54UOlWJARjKisbFvJwm+kAIUowUHRIzu5kKaZ5iZMxcMVulWtgYDXRYWR1SnOlG9JNqst1Dx8Kmkpu8eRDbtYMusL7s0ErFElyqxJN42JPEOAhAL1jh7BGI2Rrq6GJoJJzRaSwuiB4VX4U4l6/IUJMx3YqPCBzms/EzHFbvK4yXarGt5dLxZvs+DXP8g13+qcv2DPP4gjz/I4+Xy+IOU/CAlf2ZS8h0Kycv3K/8u37Noa3oXScFbS+Dn/wPcd4TLrRoBAA=="

if not os.path.exists(SAGE):
    print('Sage not found; installing Sage 10.9 via micromamba (5-15 min)...', flush=True)
    install = r"""
set -e
if [ ! -x /content/bin/bin/micromamba ]; then
  wget -q https://micro.mamba.pm/api/micromamba/linux-64/latest -O /tmp/micromamba.tar.bz2
  mkdir -p /content/bin
  tar -xjf /tmp/micromamba.tar.bz2 -C /content/bin bin/micromamba
fi
/content/bin/bin/micromamba create -y -p /content/sage-env -c conda-forge sage=10.9
"""
    subprocess.run(['bash', '-lc', install], check=True)
else:
    print('Sage already present:', SAGE, flush=True)

raw = gzip.decompress(base64.b64decode(B64)).decode('utf-8')
with open(READY, 'w', encoding='utf-8') as f:
    f.write(raw)
print('Wrote', READY, len(raw), 'chars', flush=True)
print(subprocess.check_output([SAGE, '--version']).decode().strip(), flush=True)


## Wrapper

In [ ]:
import subprocess, sys, os, time, json, shutil

SAGE = '/content/sage-env/bin/sage'
READY = '/content/LND_ready.sage'
RESULTS = '/content/LND_results'
os.makedirs(RESULTS, exist_ok=True)

try:
    SAVE_DIR = '/content/drive/MyDrive/LND_results' if os.path.isdir('/content/drive/MyDrive') else None
except Exception:
    SAVE_DIR = None


def run_pipeline(m, r, max_deg, Q_list=['X'], save_dir=None):
    """Run the notebook's full pipeline in a Sage subprocess.

    Streams every line of Sage output into this cell, saves the complete log,
    and returns (LIST1, LIST2) exactly as the notebook's run_pipeline does.
    """
    with open(READY, encoding='utf-8') as f:
        header = f.read()

    tag = f"{m}_{r}_{max_deg}"
    tail = f"""
STOP_AFTER_FILTER = False
print("### RUN m={m} r={r} max_deg={max_deg} Q_list={Q_list!r} ###", flush=True)
LIST1, LIST2 = run_pipeline({m}, {r}, {max_deg}, Q_list={Q_list!r})
print("LIST1_LEN", len(LIST1), flush=True)
for _f in LIST1:
    print("LIST1_POLY", _f, flush=True)
print("LIST2_LEN", len(LIST2), flush=True)
for _f in LIST2:
    print("LIST2_POLY", _f, flush=True)
"""

    path = f'/content/_lnd_run_{tag}.sage'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(header + tail)

    log_path = f'{RESULTS}/LND_{tag}.log'
    lines = []
    print(f'[colab] sage {path}  (log: {log_path})', flush=True)
    t0 = time.time()
    proc = subprocess.Popen([SAGE, path], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT)

    with open(log_path, 'w', encoding='utf-8') as log:
        for raw in proc.stdout:
            line = raw.decode('utf-8', 'replace')
            sys.stdout.write(line)
            sys.stdout.flush()
            log.write(line)
            log.flush()
            lines.append(line.rstrip('\n'))
    rc = proc.wait()

    print('SAGE_EXIT', rc, flush=True)
    print('WALL_SEC', round(time.time() - t0, 1), flush=True)

    list1 = [l.split('LIST1_POLY ', 1)[1] for l in lines if l.startswith('LIST1_POLY ')]
    list2 = [l.split('LIST2_POLY ', 1)[1] for l in lines if l.startswith('LIST2_POLY ')]

    with open(f'{RESULTS}/LND_{tag}.json', 'w', encoding='utf-8') as f:
        json.dump({'m': m, 'r': r, 'max_deg': max_deg, 'Q_list': Q_list,
                   'LIST1': list1, 'LIST2': list2, 'exit': rc,
                   'wall_sec': round(time.time() - t0, 1)}, f, indent=1)

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        shutil.copy(log_path, save_dir)
        shutil.copy(f'{RESULTS}/LND_{tag}.json', save_dir)
        print('[colab] saved to', save_dir, flush=True)

    return list1, list2


## Optional: mount Drive (recommended for long runs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Smoke test (fast — run this first)

In [ ]:
LIST1, LIST2 = run_pipeline(5, 3, 10, Q_list=['X'], save_dir=SAVE_DIR)


## Case: m=5, r=3, max_deg=38

In [ ]:
LIST1, LIST2 = run_pipeline(5, 3, 38, Q_list=['X'], save_dir=SAVE_DIR)


## Case: m=8, r=3, max_deg=40

In [ ]:
LIST1, LIST2 = run_pipeline(8, 3, 40, Q_list=['X'], save_dir=SAVE_DIR)


## Scaling baseline: m=8, r=5, max_deg=47 (known ~24 s locally)

In [ ]:
LIST1, LIST2 = run_pipeline(8, 5, 47, Q_list=['X'], save_dir=SAVE_DIR)


## Scaling: m=8, r=5, max_deg=55

In [ ]:
LIST1, LIST2 = run_pipeline(8, 5, 55, Q_list=['X'], save_dir=SAVE_DIR)


## Case: m=8, r=5, max_deg=60 (heaviest)

In [ ]:
LIST1, LIST2 = run_pipeline(8, 5, 60, Q_list=['X'], save_dir=SAVE_DIR)


## Notes

- The wrapper always sets `STOP_AFTER_FILTER = False`, so the full pipeline (generators + relations + verification) runs.
- `LIST1` and `LIST2` are returned as lists of polynomial strings; the cell also prints every polynomial.
- If Colab disconnects, the subprocess dies. Long cases should be split (e.g. lower `max_deg` first) and saved to Drive after each run.
- Free tier hard limit is ~12 h per session; idle disconnect is ~90 min.